In [1]:
import numpy as np
import onnx
import onnxruntime as ort

Explore the downloaded onnx model to answer Q1

## Question 1

To be able to use this model, we need to know the name of the input and output nodes.

In [2]:
onnx_model = onnx.load("./hair_classifier_v1.onnx")

In [3]:
print("input_names: ")
for inp in onnx_model.graph.input:
    print(inp.name)

input_names: 
input


In [4]:
print("output_names: ")
for outp in onnx_model.graph.output:
    print(outp.name)

output_names: 
output


What's the name of the output:

The name is output

## Preparing images

In [5]:
from io import BytesIO
from urllib import request
from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize(target_size, Image.NEAREST)
    return img

Let's download and resize this image:

In [6]:
img = download_image("https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg")

In [36]:
img.size

(1024, 1024)

In [7]:
target_size = (200, 200)
prepared_img = prepare_image(img, target_size)

In [35]:
prepared_img.size

(200, 200)

## Question 2

Based on the previous homework, what should be the target size for 
the image?

The target size for the image should be 200x200

Now we need to turn the image into numpy array and pre-process it.

In [8]:
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

def pre_process_img(img):
    img_arr = np.array(img).astype(np.float32)
    preproc_img = (img_arr / 255.0 - mean) / std
    return preproc_img


In [9]:
normalized_img = pre_process_img(prepared_img)

In [38]:
normalized_img.shape

(200, 200, 3)

## Question 3

After the pre-processing, what's the value in the first pixel, the R channel?

In [10]:
normalized_img.shape

(200, 200, 3)

In [11]:
#Red channel: array[:, :, 0]
#Green channel: array[:, :, 1]
#Blue channel: array[:, :, 2]

r_first = normalized_img[0, 0, 0]

In [12]:
print(f"Value of first pixel in red channel is: {r_first}")

Value of first pixel in red channel is: -1.0732939435925546


## Question 4

Now let's apply this model to this image. What's the output of the model?

In [14]:
#prepare normalized_img for onnx (nhcw format)

#original shape: (200, 200, 3)
#axes are: 0 = height (200), 1 = width (200), 2 = channels (3)
#HWC to CHW
tensor = np.transpose(normalized_img, (2, 0, 1))
#add batch size
input_tensor = np.expand_dims(tensor, axis=0)

In [15]:
#check shape
input_tensor.shape

(1, 3, 200, 200)

In [16]:
type(input_tensor)

numpy.ndarray

In [17]:
onnx_session = ort.InferenceSession("./hair_classifier_v1.onnx")

In [18]:
input_name = onnx_session.get_inputs()[0].name

In [19]:
outputs = onnx_session.run(None, {input_name: input_tensor.astype(np.float32)})

In [20]:
prediction = outputs[0]

In [21]:
prediction

array([[0.09156627]], dtype=float32)

In [22]:
score = prediction[0][0]

In [23]:
print(f"The output of the model is: {score}")

The output of the model is: 0.09156627207994461


## Question 5

Download the base image agrigorev/model-2025-hairstyle:v1

So what's the size of this base image?

To me the image size is ~608MB

## Question 6

Now let's extend this docker image, install all the required libraries and add the code for lambda.

You don't need to include the model in the image. It's already included. The name of the file with the model is hair_classifier_empty.onnx and it's in the current workdir in the image (see the Dockerfile above for the reference). The provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.

Now run the container locally.

Score this image: 

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

In [24]:
img = download_image("https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg")
prepared_img = prepare_image(img, target_size)
normalized_img = pre_process_img(prepared_img)
tensor = np.transpose(normalized_img, (2, 0, 1))
input_tensor = np.expand_dims(tensor, axis=0)

In [25]:
input_tensor.shape

(1, 3, 200, 200)

In [26]:
onnx_session = ort.InferenceSession("./hair_classifier_empty.onnx")

In [27]:
input_name = onnx_session.get_inputs()[0].name

In [28]:
outputs = onnx_session.run(None, {input_name: input_tensor.astype(np.float32)})

In [29]:
prediction = outputs[0]

In [30]:
score = prediction[0][0]

In [31]:
print(f"The output of the model is: {score}")

The output of the model is: -0.10220833122730255
